# E-commerce Sales Analytics

## Data Cleaning and Exploratory Data Analysis (EDA)

This notebook inspects data quality and analyzes sales by month, country, product, and customer.

**Dataset required:** `Ecommerce_Sales_Data.xlsx`

Upload the Excel file to the Colab working directory before running the analysis.

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load dataset
file_path = Path("Ecommerce_Sales_Data.xlsx")

if not file_path.exists():
    raise FileNotFoundError(
        "Ecommerce_Sales_Data.xlsx was not found. "
        "Upload the Excel file to the Colab working directory."
    )

df = pd.read_excel(file_path)

required_columns = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country", "Total Sales"
]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("DATASET OVERVIEW")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print("\nData types and non-null counts:")
df.info()
display(df.head())

DATASET OVERVIEW
Rows: 527,906
Columns: 9

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 527906 entries, 0 to 527905
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    527906 non-null  int64         
 1   StockCode    527906 non-null  object        
 2   Description  527906 non-null  object        
 3   Quantity     527906 non-null  int64         
 4   InvoiceDate  527906 non-null  datetime64[ns]
 5   UnitPrice    527906 non-null  float64       
 6   CustomerID   396470 non-null  float64       
 7   Country      527906 non-null  object        
 8   Total Sales  527906 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 36.2+ MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Total Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-01-12 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-01-12 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-01-12 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-01-12 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-01-12 08:26:00,3.39,17850.0,United Kingdom,20.34


## 1. Data Quality Checks

Review missing values, duplicate rows, and negative values before deciding whether any records should be removed or changed.

In [11]:
missing_summary = df.isna().sum().to_frame(name="Missing Count")
missing_summary["Missing Percentage (%)"] = (
    missing_summary["Missing Count"] / len(df) * 100
).round(2)
display(missing_summary)

duplicate_count = int(df.duplicated().sum())
print(f"Duplicate rows detected: {duplicate_count:,}")

print("\nNegative values:")
for col in ["Quantity", "UnitPrice", "Total Sales"]:
    negative_count = (pd.to_numeric(df[col], errors="coerce") < 0).sum()
    print(f"{col}: {negative_count:,}")

# Missing values are not automatically replaced with zero.
# Duplicate rows are not automatically deleted.

,Missing Count,Missing Percentage (%)
InvoiceNo,0,0.0
StockCode,0,0.0
Description,0,0.0
Quantity,0,0.0
InvoiceDate,0,0.0
UnitPrice,0,0.0
CustomerID,131436,24.9
Country,0,0.0
Total Sales,0,0.0


Duplicate rows detected: 5,221

Negative values:
Quantity: 0
UnitPrice: 0
Total Sales: 0


## 2. Data Cleaning and Feature Engineering

In [12]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

for col in ["Quantity", "UnitPrice", "Total Sales", "CustomerID"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["InvoiceNo"] = df["InvoiceNo"].astype("string").str.strip()
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype("string")
df["IsCancellation"] = df["InvoiceNo"].str.startswith("C", na=False)

print("Data types after conversion:")
print(df.dtypes)
print(f"\nCancellation invoice rows: {df['IsCancellation'].sum():,}")
print(f"Rows with missing CustomerID: {df['CustomerID'].isna().sum():,}")

Data types after conversion:
InvoiceNo         string[python]
StockCode                 object
Description               object
Quantity                   int64
InvoiceDate       datetime64[ns]
UnitPrice                float64
CustomerID               float64
Country                   object
Total Sales              float64
YearMonth         string[python]
IsCancellation           boolean
dtype: object

Cancellation invoice rows: 0
Rows with missing CustomerID: 131,436


## 3. Key Business Metrics

Revenue is calculated from `Total Sales`. AOV is defined here as total net sales (including negative adjustments present in the data) divided by the number of unique invoices not beginning with `C`. Interpret this definition when presenting results.

In [13]:
valid_sales = df.dropna(subset=["InvoiceNo", "Total Sales"]).copy()

total_revenue = valid_sales["Total Sales"].sum()
sales_invoices = valid_sales.loc[
    ~valid_sales["IsCancellation"], "InvoiceNo"
].nunique()

aov = total_revenue / sales_invoices if sales_invoices else np.nan

print(f"Total Revenue (net): {total_revenue:,.2f}")
print(f"Unique Sales Invoices: {sales_invoices:,}")
print(f"Average Order Value (AOV): {aov:,.2f}")
print(f"Unique Customers: {df['CustomerID'].nunique():,}")
print(f"Unique Products: {df['StockCode'].nunique():,}")
print(f"Countries: {df['Country'].nunique():,}")

Total Revenue (net): 10,278,874.89
Unique Sales Invoices: 19,776
Average Order Value (AOV): 519.77
Unique Customers: 4,334
Unique Products: 3,911
Countries: 38


## 4. Monthly Sales Analysis

In [14]:
monthly_sales = (
    valid_sales.dropna(subset=["YearMonth"])
    .groupby("YearMonth", as_index=False)
    .agg(
        Total_Sales=("Total Sales", "sum"),
        Invoice_Count=("InvoiceNo", "nunique")
    )
    .sort_values("YearMonth")
)

display(monthly_sales)

if not monthly_sales.empty:
    best_month = monthly_sales.loc[monthly_sales["Total_Sales"].idxmax()]
    print(f"Highest sales month: {best_month['YearMonth']}")
    print(f"Sales in that month: {best_month['Total_Sales']:,.2f}")

,YearMonth,Total_Sales,Invoice_Count
0,2010-01,57676.33,127
1,2010-02,47715.38,141
2,2010-03,44833.98,68
3,2010-05,31318.99,88
4,2010-06,51973.43,102
5,2010-07,81519.34,82
6,2010-08,44308.60,116
7,2010-09,50130.51,106
8,2010-10,56850.03,76
9,2010-12,312281.77,645


Highest sales month: 2011-11
Sales in that month: 1,286,204.92


## 5. Sales by Country

In [15]:
country_sales = (
    valid_sales.groupby("Country", as_index=False)
    .agg(
        Total_Sales=("Total Sales", "sum"),
        Invoice_Count=("InvoiceNo", "nunique"),
        Unique_Customers=("CustomerID", "nunique")
    )
    .sort_values("Total_Sales", ascending=False)
)

display(country_sales.head(10))

,Country,Total_Sales,Invoice_Count,Unique_Customers
36,United Kingdom,8749427.49,17901,3916
24,Netherlands,283889.34,93,9
10,EIRE,276404.30,284,3
14,Germany,205569.89,443,94
13,France,184768.74,383,87
0,Australia,138171.31,56,9
31,Spain,55725.11,88,30
33,Switzerland,53087.90,50,21
20,Japan,37416.37,19,8
3,Belgium,36927.34,98,25


## 6. Top Products by Sales

In [16]:
product_sales = (
    valid_sales.groupby(
        ["StockCode", "Description"],
        dropna=False,
        as_index=False
    )
    .agg(
        Total_Sales=("Total Sales", "sum"),
        Total_Quantity=("Quantity", "sum")
    )
    .sort_values("Total_Sales", ascending=False)
)

display(product_sales.head(10))

,StockCode,Description,Total_Sales,Total_Quantity
1340,22423,REGENCY CAKESTAND 3 TIER,174484.74,13879
2668,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
3640,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104340.29,37599
2877,47566,PARTY BUNTING,99504.33,18295
3619,85099B,JUMBO BAG RED RETROSPOT,94340.05,48474
2123,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033
2029,23084,RABBIT NIGHT LIGHT,66964.99,30788
1022,22086,PAPER CHAIN KIT 50'S CHRISTMAS,64952.29,19355
3416,84879,ASSORTED COLOUR BIRD ORNAMENT,59094.93,36461
3059,79321,CHILLI LIGHTS,54117.76,10306


## 7. Customer Analysis

In [17]:
customer_sales = (
    valid_sales.dropna(subset=["CustomerID"])
    .groupby("CustomerID", as_index=False)
    .agg(
        Total_Sales=("Total Sales", "sum"),
        Invoice_Count=("InvoiceNo", "nunique")
    )
    .sort_values("Total_Sales", ascending=False)
)

display(customer_sales.head(10))

,CustomerID,Total_Sales,Invoice_Count
1689,14646.0,279138.02,72
4197,18102.0,259657.30,60
3725,17450.0,194550.79,46
3006,16446.0,168472.50,2
1879,14911.0,140450.72,199
55,12415.0,124564.53,20
1333,14156.0,117379.63,55
3768,17511.0,91062.38,31
0,12346.0,77183.60,1
2700,16029.0,72882.09,62


## 8. Final Data Quality Summary

Review these checks before deciding whether to remove duplicates or impute missing data.

In [18]:
print(f"Rows in dataset: {len(df):,}")
print(f"Duplicate rows detected: {duplicate_count:,}")
print(f"Rows with missing InvoiceDate: {df['InvoiceDate'].isna().sum():,}")
print(f"Rows with missing Total Sales: {df['Total Sales'].isna().sum():,}")
print(f"Rows with missing CustomerID: {df['CustomerID'].isna().sum():,}")

print("\nData cleaning and EDA completed.")
print("Review data quality findings before removing or imputing data.")

Rows in dataset: 527,906
Duplicate rows detected: 5,221
Rows with missing InvoiceDate: 0
Rows with missing Total Sales: 0
Rows with missing CustomerID: 131,436

Data cleaning and EDA completed.
Review data quality findings before removing or imputing data.
